In [1]:
!pip install transformers datasets torch numpy tqdm

In [2]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm
import json

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = AutoModel.from_pretrained(
    "JayShah07/tinybert-dual-classifier"
).to(device)

encoder.eval()

tokenizer = AutoTokenizer.from_pretrained(
    "JayShah07/tinybert-dual-classifier"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/57.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [4]:
dataset = load_dataset("JayShah07/reporting_final_dataset")
train_ds = dataset["train"]

print("Train size:", len(train_ds))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/47.8k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3097 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/387 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/388 [00:00<?, ? examples/s]

Train size: 3097


In [5]:
embeddings = []
confidences = []

softmax = torch.nn.Softmax(dim=-1)

for sample in tqdm(train_ds):
    text = sample["query"]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

        # ----- confidence proxy (entropy / max softmax) -----
        pooled = cls_embedding
        logits_proxy = pooled @ pooled.T  # stable proxy
        probs = softmax(logits_proxy)
        confidences.append(probs.max().item())

100%|██████████| 3097/3097 [02:53<00:00, 17.86it/s]


In [6]:
embeddings = np.array(embeddings)
confidences = np.array(confidences)

print("Embeddings shape:", embeddings.shape)
print("Confidence samples:", confidences[:5])

Embeddings shape: (3097, 312)
Confidence samples: [1. 1. 1. 1. 1.]


In [7]:
embedding_baseline = embeddings.mean(axis=0)
embedding_cov = np.cov(embeddings.T)

np.save("embedding_baseline.npy", embedding_baseline)
np.save("embedding_cov.npy", embedding_cov)
np.save("confidence_baseline.npy", confidences)

In [8]:
hist, bins = np.histogram(confidences, bins=10)

with open("confidence_hist.json", "w") as f:
    json.dump({
        "hist": hist.tolist(),
        "bins": bins.tolist()
    }, f)